In [23]:
import os, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np, kagglehub
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from aijack.collaborative.fedavg import FedAVGClient, FedAVGServer
from aijack.defense.dp.manager.dp_manager import DPSGDManager
from aijack.defense.dp.manager.accountant import GeneralMomentAccountant
from aijack.attack.inversion import GradientInversion_Attack

In [24]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
batch = 32
lat_dim = 100
clients = 3

s = 0.5           # Rumore aggiunto (calibrato per utilità/privacy)
l2= 1.0    # Limite alla sensibilità del singolo gradiente
t_delta = 1e-5   # Probabilità di leak ammessa

In [25]:
def prepare_federated_data(num_clients=clients, batch_size=batch):
    path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
    data_dir = os.path.join(path, 'COVID-19_Radiography_Dataset')

        # Modifica la trasformazione per assicurare 3 canali se vuoi mantenere il default della classe
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.Lambda(lambda x: x.convert('RGB')), # Forza 3 canali
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
    ])

    full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    indices = list(range(len(full_dataset)))
    np.random.shuffle(indices)

    split = len(full_dataset) // num_clients
    loaders = []
    for i in range(num_clients):
        subset = Subset(full_dataset, indices[i*split : (i+1)*split])
        loaders.append(DataLoader(subset, batch_size=batch_size, shuffle=True))
    return loaders, full_dataset

client_loaders, full_dataset = prepare_federated_data()

In [26]:
# =================================================================
# 2. ARCHITETTURA GENERATIVE ADVERSARIAL NETWORK (DCGAN)
# =================================================================

class Generator(nn.Module):
    def __init__(self, latent_dim=lat_dim, img_channels=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z): return self.main(z)

class Discriminator(nn.Module):
    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(img_channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, img): return self.main(img).view(-1, 1)


global_generator = Generator().to(device)
local_discriminators = [Discriminator().to(device) for _ in range(clients)]


gen_opt = optim.Adam(global_generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
dis_opts = [optim.Adam(d.parameters(), lr=0.0002, betas=(0.5, 0.999)) for d in local_discriminators]

In [27]:
# 1. Inizializza l'Accountant per monitorare il consumo di privacy (Epsilon)
accountant = GeneralMomentAccountant(
    noise_type="Gaussian",
    search="ternary",
    precision=0.001,
    bound_type="rdp_upperbound_closedformula",
    backend="python"
)

# 2. Crea il DPSGDManager
# Questo oggetto gestirà il clipping dei gradienti e l'aggiunta del rumore
privacy_manager = DPSGDManager(
    accountant,
    optim.Adam, # Tipo di ottimizzatore da "privatizzare"
    l2_norm_clip=l2,
    dataset=full_dataset,
    lot_size=batch,
    batch_size=batch,
    iterations=100 # Numero di iterazioni pianificate
)

In [28]:
from torch.utils.data import TensorDataset

# 1. Privatizziamo l'ottimizzatore per il primo discriminatore
# noise_multiplier è il nostro parametro 's' (0.5)
dpoptimizer_cls, lot_loader, batch_loader = privacy_manager.privatize(noise_multiplier=s)

# 2. Creiamo l'istanza dell'ottimizzatore DP per il Client 0
optimizer_d0 = dpoptimizer_cls(local_discriminators[0].parameters(), lr=0.0002)

# --- IMPORTANTE: Inizializzazione dei Gradienti ---
# Eseguiamo un "passaggio a vuoto" per evitare l'AttributeError: 'NoneType' object has no attribute 'data'
# Questo crea gli oggetti .grad necessari all'accountant prima del loop reale.
dummy_x, _ = next(iter(client_loaders[0]))
dummy_x = dummy_x.to(device)
local_discriminators[0].zero_grad()
out = local_discriminators[0](dummy_x)
out.mean().backward()
optimizer_d0.zero_grad()

print("🛡️ Ottimizzatore privatizzato e gradienti inizializzati correttamente.")

🛡️ Ottimizzatore privatizzato e gradienti inizializzati correttamente.


In [29]:
epochs = 10
criterion = nn.BCELoss()

for epoch in range(epochs):
    # Iteriamo sui "Lots" (gruppi di batch per calibrare il rumore DP)
    for X_lot, y_lot in lot_loader(optimizer_d0):
        # Iteriamo sui singoli Batch all'interno del Lot
        for X_batch, y_batch in batch_loader(TensorDataset(X_lot, y_lot)):
            X_batch = X_batch.to(device)
            b_size = X_batch.size(0)
            
            # --- 1. ADDESTRAMENTO DISCRIMINATORE (DP-SGD) ---
            optimizer_d0.zero_grad()
            
            # Immagini reali
            output_real = local_discriminators[0](X_batch)
            loss_real = criterion(output_real, torch.ones(b_size, 1).to(device))
            
            # Immagini sintetiche
            noise = torch.randn(b_size, lat_dim, 1, 1).to(device)
            fake_images = global_generator(noise)
            output_fake = local_discriminators[0](fake_images.detach())
            loss_fake = criterion(output_fake, torch.zeros(b_size, 1).to(device))
            
            # Backpropagation (il manager gestirà il clipping e il rumore qui)
            loss_d = loss_real + loss_fake
            loss_d.backward()
            optimizer_d0.step()
            
            # --- 2. ADDESTRAMENTO GENERATORE (Standard) ---
            gen_opt.zero_grad()
            output_g = local_discriminators[0](fake_images)
            loss_g = criterion(output_g, torch.ones(b_size, 1).to(device))
            loss_g.backward()
            gen_opt.step()

    # Calcolo dell'Epsilon attuale (budget di privacy speso)
    current_eps = accountant.get_epsilon(delta=t_delta)
    print(f"Epoch [{epoch+1}/{epochs}] - Epsilon: {current_eps:.2f} - Loss D: {loss_d.item():.4f} - Loss G: {loss_g.item():.4f}")

AttributeError: 'NoneType' object has no attribute 'data'